# Inferencia en vivo con Random Forest y MediaPipe
Este notebook permite probar el modelo entrenado en un video nuevo o en la webcam, extrayendo features por ventana y mostrando la predicción de actividad en tiempo real.

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from collections import deque

# Cargar modelo y mapeo de etiquetas
model_path = Path('../results/random_forest_model.joblib')
clf = joblib.load(model_path)

import json
with open('../results/random_forest_metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)
label_names = list(metrics.keys())[:-3]  # Quita avg/accuracy

# Funciones auxiliares
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
    return np.degrees(angle)

def calculate_distance(p1, p2):
    p1 = np.array(p1)
    p2 = np.array(p2)
    return np.linalg.norm(p1 - p2)

## Selección de fuente de video
Puedes cambiar `video_source` a 0 para webcam, o a la ruta de un archivo de video.

In [ ]:
# Cambia a 0 para webcam, o a la ruta de un video
video_source = 0  # o 'ruta/a/tu/video.mp4'
cap = cv2.VideoCapture(video_source)
target_fps = 30
window_size = int(0.5 * target_fps)
step_size = int(0.25 * target_fps)
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5)

frame_buffer = deque(maxlen=window_size)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    # Dibujar landmarks en el frame
    annotated_frame = frame.copy()
    if results.pose_landmarks:
        mp.solutions.drawing_utils.draw_landmarks(
            annotated_frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        lm = results.pose_landmarks.landmark
        coords = {
            'hip': (lm[23].x, lm[23].y),
            'knee': (lm[25].x, lm[25].y),
            'ankle': (lm[27].x, lm[27].y),
            'shoulder': (lm[11].x, lm[11].y)
        }
        frame_buffer.append(coords)
    else:
        frame_buffer.append(None)
    # Solo predecir si hay suficientes frames
    if len(frame_buffer) == window_size and frame_buffer.count(None) < window_size * 0.3:
        window_valid = [f for f in frame_buffer if f is not None]
        hips = np.array([f['hip'] for f in window_valid])
        knees = np.array([f['knee'] for f in window_valid])
        ankles = np.array([f['ankle'] for f in window_valid])
        shoulders = np.array([f['shoulder'] for f in window_valid])
        knee_angles = [calculate_angle(hip, knee, ankle) for hip, knee, ankle in zip(hips, knees, ankles)]
        trunk_incl = [np.degrees(np.arctan2(s[0]-h[0], h[1]-s[1])) for s, h in zip(shoulders, hips)]
        dist_sh_hip = [calculate_distance(s, h) for s, h in zip(shoulders, hips)]
        feats = np.array([[
            np.mean(knee_angles), np.std(knee_angles),
            np.mean(trunk_incl), np.std(trunk_incl),
            np.mean(dist_sh_hip), np.std(dist_sh_hip)
        ]])
        pred = clf.predict(feats)[0]
        pred_label = label_names[pred] if pred < len(label_names) else str(pred)
        cv2.putText(annotated_frame, f'Pred: {pred_label}', (30, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow('Actividad y Landmarks', annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()